# Mini-Project 4: Rainfall Pattern & Crop Suitability Analyser
### Agronomic Assessment Matrix — Kampala, Gulu, and Mbarara

In [ ]:
import numpy as np
import pandas as pd
from src.rainfall_analyser import Region, CropRule, ClimateMetrics

# Illustrative Data Profiles
k_rain = [120, 140, 180, 200, 220, 180, 90, 70, 60, 100, 110, 130]
g_rain = [8, 25, 75, 160, 190, 145, 170, 215, 175, 150, 60, 15]
m_rain = [70, 85, 120, 140, 90, 25, 20, 55, 100, 125, 120, 90]

regions = [
    Region("Kampala", k_rain),
    Region("Gulu", g_rain),
    Region("Mbarara", m_rain)
]

for r in regions:
    print(f"{r.name} -> Total: {r.total_annual}mm | CV: {r.coefficient_of_variation:.3f} | Climate Regime: {r.classify_regime()}")

In [ ]:
# Crop Requirements Definitions (Based on FAO parameters)
rules = [
    CropRule("Maize", 60, 150),  # Maize threshold
    CropRule("Beans", 50, 120),  # Beans threshold
    CropRule("Coffee", 90, 200)  # Perennial Arabica/Robusta threshold
]

months = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
matrix_data = []

for r in regions:
    for m_idx, m_name in enumerate(months):
        rain = r.rainfall[m_idx]
        status = "Suboptimal"
        for rule in rules:
            res = rule.assess_month(rain)
            if "Good" in res:
                status = res
                break
        if rain < 40: status = "Drought Risk"
        if rain > 250: status = "Waterlogging Risk"
        matrix_data.append({"Region": r.name, "Month": m_name, "Rainfall": rain, "Suitability": status})

df_suit = pd.DataFrame(matrix_data)
print(df_suit.head(12))

In [ ]:
# Generate Matrix Comparisons between Region Vectors
names = [r.name for r in regions]
cos_matrix = np.zeros((3, 3))

for i, r1 in enumerate(regions):
    for j, r2 in enumerate(regions):
        cos_matrix[i, j] = ClimateMetrics.manual_cosine_similarity(r1.rainfall, r2.rainfall)

df_cos = pd.DataFrame(cos_matrix, index=names, columns=names)
print("\n--- Cosine Similarity Matrix ---")
print(df_cos)

### Written Project Analysis: Findings & Limitations
Using math.cos() for vector assessment is a fundamental error; math.cos() computes the cosine of a singular radian angle rather than evaluating directional similarities between multi-dimensional arrays. Correcting this with true vector math shows high cosine similarities between regions like Kampala and Mbarara due to their directional distribution shapes. Cosine similarity isolates vector direction while completely disregarding magnitude. Therefore, a very dry region can register as highly 'similar' to a wet region if their relative seasonal rises and falls occur during the same calendar months.

Peak detection using scipy algorithms correctly classifies Kampala and Mbarara as bimodal regimes, mirroring the equatorial movement across central and southern Uganda. Conversely, Gulu exhibits a distinct unimodal profile characteristic of northern sub-Saharan zones. A key limitation of this analysis is that it uses a static monthly rainfall total, which fails to track sub-month dry spells or daily soil percolation behavior essential for accurate seed planting schedules.

### 🌾 Extension: Agricultural Extension Advisory Note (Gulu District Focus)
To maximize seasonal yields and minimize crop failure in the Gulu region, farmers should realign their planting timelines to match the area's strict unimodal climate profile. Northern Uganda experiences a severe dry season from December through February, presenting a critical drought risk where monthly precipitation falls below 30mm. Planting during this window will lead to complete crop failure.

The rainy season reliably starts in April and peaks between July and September (reaching over 200mm). Farmers should prepare their fields in March and plant **Maize** and **Beans** at the start of the rains in **April**. This timeline ensures that the crops' peak water needs align with the heavy rains of May and June, while allowing for a dry harvest period in late November. For high-value perennial options like **Coffee**, planting should occur exclusively in **May** to allow root networks to establish before the dry season begins in December.